In [1]:
import praw
import boto3
import os

from better_profanity import profanity
from moviepy import *

from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload

profanity.load_censor_words()

MAX_COMMENT_LENGTH = 200

In [2]:
# Setup
polly = boto3.client(
    'polly',
    aws_access_key_id=os.environ.get('AWS_POLLY_ACCESS_KEY'),
    aws_secret_access_key=os.environ.get('AWS_POLLY_SECRET_ACCESS_KEY'),
    region_name='us-east-1'
)
reddit = praw.Reddit(
    client_id=os.getenv("REDDIT_CLIENT_ID"),
    client_secret=os.getenv("REDDIT_CLIENT_SECRET"),
    user_agent=os.getenv("REDDIT_USER_AGENT")
)
subreddit = reddit.subreddit("AskReddit")

In [ ]:
# Parse reddit content
top_post = next(
    (
        post 
        for post in subreddit.top(time_filter='day', limit=10) 
        if (
            not post.over_18 
            and not profanity.contains_profanity(post.title)
            and not len(post.title) > 75
        )
    ),
    None
)

if top_post:
    top_post.comment_sort = "top"  # Sort by top comments
    top_post.comments.replace_more(limit=0)  # Replace "load more" placeholders

    # Filter and select top comments within the length limit
    top_comments = [
        comment for comment in top_post.comments 
        if len(comment.body) <= MAX_COMMENT_LENGTH
        # TODO: Consider nsfw comment filtering?
    ][:3] 

    # Check if there are enough valid comments
    if len(top_comments) < 3:
        raise ValueError("Not enough valid comments found for video creation.")
else:
    raise ValueError("No suitable Reddit post found.")

if top_post and top_comments:
    reddit_title = top_post.title
    top_comment_0 = top_comments[0].body
    top_comment_1 = top_comments[1].body
    top_comment_2 = top_comments[2].body
    reddit_link = top_post.shortlink

    print("Selected post:", reddit_title)
    print(f"Post link: {reddit_link}")
    print(f"top_comment_0: {top_comment_0}")
    print(f"top_comment_1: {top_comment_1}")
    print(f"top_comment_2: {top_comment_2}")

In [ ]:
# Generate title audio file
try:
    title_resp = polly.synthesize_speech(
        Text=reddit_title,
        OutputFormat='mp3',
        VoiceId='Salli'
    )

    with open('assets/title_audio.mp3', 'wb') as f:
        f.write(title_resp['AudioStream'].read())
    print(f"Saved title_audio.mp3")

    # Generate comment audio files
    for i, comment in enumerate(top_comments, 1):
        response = polly.synthesize_speech(
            Text=comment.body,
            OutputFormat='mp3',
            VoiceId='Salli'
        )
        filename = f"assets/comment_{i}.mp3"
        with open(filename, 'wb') as f:
            f.write(response['AudioStream'].read())
        print(f"Saved {filename}")
except Exception as e:
    raise ValueError(f"Failed to generate audio files: {e}.")

In [ ]:
def create_segment_clip(bg_img_path, audio_path, text=None):
    """
    Returns a MoviePy clip of 'background_path' with
    overlay text matching 'audio_path' duration.
    """
    # 1) Load the audio
    audio_clip = AudioFileClip(audio_path)
    audio_duration = audio_clip.duration

    # 2) Load the background img
    bg_clip = ImageClip(bg_img_path)

    # 3) Create a TextClip to overlay
    if text:
        try:
            # For GH action, need dif font
            text_clip = TextClip(
                text=text,
                font_size=60,
                color="#ff5d01",
                # 'caption' mode wraps text automatically to fit the size
                method='caption',
                size=(1000, 1000),
                font='Lato-Bold',
                text_align='center'           
            ).with_duration(audio_duration).with_position(("center", 100))
        except Exception:
            # For local 
            text_clip = TextClip(
                text=text,
                font_size=60,
                color="#ff5d01",
                # 'caption' mode wraps text automatically to fit the size
                method='caption',
                size=(1000, 1000),
                font='Arial',
                text_align='center'           
            ).with_duration(audio_duration).with_position(("center", 100))

        # 4) Combine background + text overlay
        composite = CompositeVideoClip([bg_clip, text_clip]).with_duration(audio_duration)
    else:
        composite = CompositeVideoClip([bg_clip]).with_duration(audio_duration)

    # 5) Set the audio
    composite = composite.with_audio(audio_clip)

    return composite

def create_silent_gap(duration=0.5, background_path="assets/bg.jpg"):
    """
    Returns a silent color clip (default: 1s black screen).
    """
    return ImageClip(background_path).with_duration(duration)

try:
    title_clip = create_segment_clip(
        bg_img_path="assets/bg.jpg",
        text=reddit_title,
        audio_path="assets/title_audio.mp3"
    )

    # Comment clips
    comment_clips = []
    for i, c_text in enumerate([top_comment_0, top_comment_1, top_comment_2], 1):
        clip = create_segment_clip(
            bg_img_path="assets/bg.jpg",
            text=c_text,
            audio_path=f"assets/comment_{i}.mp3"
        )
        comment_clips.append(clip)

    question_clip = create_segment_clip(
        bg_img_path="assets/bg.jpg",
        audio_path="assets/question.mp3"
    )

    top_responses_clip = create_segment_clip(
        bg_img_path="assets/bg.jpg",
        audio_path="assets/top_responses.mp3"
    )

    gap_clip = create_silent_gap()

    # --- 4. Concatenate all clips (title + comments) into one final video ---
    # Build final sequence: [title, gap, comment 1, gap, comment 2, gap, comment 3]
    all_clips = []
    all_clips.append(question_clip)
    all_clips.append(gap_clip)
    all_clips.append(title_clip)
    all_clips.append(gap_clip)
    all_clips.append(top_responses_clip)
    all_clips.append(gap_clip)

    for comment_clip in comment_clips:
        all_clips.append(comment_clip)
        all_clips.append(gap_clip)
    final_clip = concatenate_videoclips(all_clips, method="compose")

    # --- 5. Add background music ---
    background_music = AudioFileClip("assets/funk_bg_lower.mp3").with_duration(final_clip.duration)
    composite_audio = CompositeAudioClip([final_clip.audio, background_music])
    final_clip = final_clip.with_audio(composite_audio)

    # --- 6. Export the final video ---
    final_clip.write_videofile("assets/final_askreddit_video.mp4", fps=30)
except Exception as e:
    raise ValueError(f"Failed to generate video clips: {e}.")


In [ ]:
from dotenv import load_dotenv
load_dotenv()

def authenticate_youtube():
    print(os.environ["YOUTUBE_REFRESH_TOKEN"])
    creds = Credentials(
        None,
        refresh_token=os.environ['YOUTUBE_REFRESH_TOKEN'],
        token_uri="https://oauth2.googleapis.com/token",
        client_id=os.environ['YOUTUBE_CLIENT_ID'],
        client_secret=os.environ['YOUTUBE_CLIENT_SECRET']
    )

    # Refresh the token if necessary
    if not creds.valid:
        creds.refresh(Request())
    
    return creds

# --- 8. Upload to YouTube ---
def upload_to_youtube(file_path, title, description, category_id="22", keywords=None, privacy_status="public"):
    """
    Uploads a video to YouTube.

    :param file_path: Path to the video file.
    :param title: Title of the video.
    :param description: Description of the video.
    :param category_id: YouTube video category ID (default is "People & Blogs").
    :param keywords: List of keywords/tags.
    :param privacy_status: "public", "private", or "unlisted".
    """
    creds = authenticate_youtube()
    youtube = build('youtube', 'v3', credentials=creds)

    try:
        body = {
            'snippet': {
                'title': title,
                'description': description,
                'tags': keywords if keywords else [],
                'categoryId': category_id
            },
            'status': {
                'privacyStatus': privacy_status
            }
        }

        # Upload the video file
        media_body = MediaFileUpload(file_path, chunksize=-1, resumable=True, mimetype='video/mp4')

        request = youtube.videos().insert(
            part=','.join(body.keys()),
            body=body,
            media_body=media_body
        )

        response = None
        while response is None:
            status, response = request.next_chunk()
            if status:
                print(f"Uploading... {int(status.progress() * 100)}%")

        print(f"Upload Complete! Video ID: {response['id']}")
        return response['id']

    except HttpError as e:
        print(f"An HTTP error {e.resp.status} occurred:\n{e.content}")
        return None

try:
    # Define video metadata
    video_title = f"AskReddit Daily: {reddit_title} #shorts"
    video_description = (
        f"Today's top AskReddit post: {reddit_title}\n\n"
        f"Top Comments:\n1. {top_comment_0}\n2. {top_comment_1}\n3. {top_comment_2}\n\n"
        f"{reddit_link}"
        "#AskReddit #RedditDaily #Shorts"
    )
    video_keywords = ["AskReddit", "RedditDigest", "Shorts", "RedditDaily", "Reddit"]

    # Upload the video
    video_id = upload_to_youtube(
        file_path="assets/final_askreddit_video.mp4",
        title=video_title,
        description=video_description,
        category_id="22",  # People & Blogs
        keywords=video_keywords,
        privacy_status="public"  # Change to 'private' or 'unlisted' if desired
    )
    if video_id:
        print(f"Video successfully uploaded! Watch it at https://www.youtube.com/watch?v={video_id}")
except Exception as e:
    raise ValueError(f"Failed upload. {e}")